# Model Random Forest – prognoza popytu dla sklepów

Notebook trenuje model Random Forest na zbiorze `Dane_sklepy.csv` (wynik notebooka 03), ocenia go na danych z 2025 roku i zapisuje prognozy do `wyniki/prognozy/prognozy_RF_2025.csv`. Plik ten wykorzystywany jest w notebooku 06 do porównania z modelem XGBoost.

Aby zainstalować wymagane biblioteki, należy w terminalu wpisać `pip install -r requirements.txt`.

### Załadowanie bibliotek oraz wczytanie wcześniej przygotowanego zbioru danych 

In [1]:
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Wczytanie danych 
df = pd.read_csv(r'../dane/przetworzone/Dane_sklepy.csv', sep=';',decimal=',',encoding='utf-8-sig')

### Kodowanie zmiennych tekstowych

Zmienne tekstowe, takie jak klient oraz podkategoria produktu, zostały zamienione na wartości liczbowe. Jest to wymagane, ponieważ implementacja Random Forest w scikit-learn pracuje wyłącznie na danych numerycznych.
Dla każdej zmiennej kategorii przypisano unikalny kod liczbowy za pomocą `cat.codes`. Słownik `mapowania` zapamiętuje przypisanie kodów, aby na końcu przywrócić oryginalne nazwy klientów i podkategorii.

In [2]:
mapowania = {}
kolumny_tekstowe = ['Klient', 'Podkategoria']  

for col in kolumny_tekstowe:
    cat = df[col].astype('category')
    mapowania[col] = dict(enumerate(cat.cat.categories))
    df[col] = cat.cat.codes

print(df[['Klient', 'Podkategoria']].head())

   Klient  Podkategoria
0       0             0
1       0             0
2       0             0
3       0             0
4       0             0


### Sortowanie, ustawienie indexu (data), podział zbioru danych

Dane zostały posortowane względem daty oraz podkategorii produktu. Kolejno ustawiono `data_sprzedaży` jako indeks, aby umożliwić chronologiczny podział danych (bez losowego mieszania – zapobiega to wyciekowi informacji z przyszłości).

**Podział danych na zbiory**
* Zbiór treningowy (2021–2023) – służy do nauki zależności na danych historycznych,
* Zbiór walidacyjny (2024) – służy do monitorowania jakości modelu,
* Zbiór testowy (2025) – zbiór końcowy do oceny skuteczności modelu na danych, których model nie widział.

In [3]:
# Sortowanie i ustawienie indexu
df = df.sort_values(["data_sprzedaży","Podkategoria"])
df = df.set_index('data_sprzedaży')

df_model = df

# Podział zbioru danych
train = df_model.loc['2021-01-01':'2023-12-31']   # rok treningu
val   = df_model.loc['2024-01-01':'2024-12-31']   # rok walidacji
test  = df_model.loc['2025-01-01':'2025-12-31']   # rok testu 

print(f"Rozmiar zbiorów:")
print(f"Train: {train.shape[0]:,} wierszy")
print(f"Val: {val.shape[0]:,} wierszy")
print(f"Test: {test.shape[0]:,} wierszy")

Rozmiar zbiorów:
Train: 290,175 wierszy
Val: 96,990 wierszy
Test: 96,725 wierszy


### Grupowanie danych na zmienne cechy i cel

Zmienną celu, czyli prognozowaną wartością, jest `Ilość` (dzienna sprzedaż w kg).
Pozostałe zmienne z listy `FEATURES` (20 cech: klient, podkategoria, cechy cykliczne i kalendarzowe, flagi sezonowe oraz `lag_364`) służą jako zmienne opisujące zjawisko sprzedaży.

Na tej podstawie przygotowano macierze cech `X` i wektory celu `y` dla zbioru treningowego, walidacyjnego i testowego.

In [4]:
FEATURES = [
    'Klient', 'Podkategoria',
    # Sezonowość cykliczna
    'miesiac_sin', 'miesiac_cos',
    'kwartał_sin', 'kwartał_cos',
    'day_sin', 'day_cos',
    'tydzien_roku', 'rok',
    # Cechy kalendarzowe
    'czy_niedziela', 'okres_wielkanocny', 'okres_wigilijny',
    'czy_ferie_slask', 'czy_wakacje', 'czy_majowka',
    'czy_swieto', 'dni_do_swieta',
    'czy_szczyt_sklep',
    # Lagi 
    'lag_364']

X_train = train[FEATURES]
y_train = train['Ilość']

X_val = val[FEATURES]
y_val = val['Ilość']

X_test = test[FEATURES]
y_test = test['Ilość']

### Definiowanie funkcji do oceny jakości modelu
Funkcja `print_metrics` wylicza:
* **MAE** – średni błąd bezwzględny [kg],
* **RMSE** – pierwiastek błędu średniokwadratowego [kg], silniej karze duże pomyłki,
* **R²** – jaka część zmienności sprzedaży jest wyjaśniona przez model,
* **Bias** – średnia różnica *prognoza − rzeczywistość*; wartość ujemna oznacza niedoszacowanie popytu,
* **MAE / średnia** – błąd w odniesieniu do średniej sprzedaży,
* **WMAPE** – ważony procentowy błąd prognozy, liczony tylko dla dni ze sprzedażą większą od zera (dni zerowe zaburzają tę statystykę).

In [5]:
# Definiowanie funkcji do oceny modelu 
def wmape(y_fact, y_pred):
    # Wykluczenie wartości ponieważ zaburza statystykę 
    mask = y_fact > 0
    return np.sum(np.abs(y_fact[mask] - y_pred[mask])) / np.sum(y_fact[mask]) * 100

def print_metrics(name, y_fact, y_pred):

    mae  = mean_absolute_error(y_fact, y_pred)
    rmse = np.sqrt(mean_squared_error(y_fact, y_pred))
    r2   = r2_score(y_fact, y_pred)

    bias = np.mean(y_pred - y_fact)

    mean_ = y_fact.mean()
    wm    = wmape(y_fact.values, y_pred)

    print(f"\n{'─'*20}")
    print(f"  {name}")
    print(f"{'─'*20}")
    print(f"MAE           : {mae:.4f} kg")
    print(f"RMSE          : {rmse:.4f} kg")
    print(f"R²            : {r2:.4f}")
    print(f"Bias          : {bias:.4f} kg")
    print(f"Średnia       : {mean_:.4f} kg")
    print(f"MAE / średnia : {mae/mean_*100:.1f}%")
    print(f"WMAPE         : {wm:.1f}%")

    return mae, rmse, r2, bias

### Definiowanie parametrów modelu Random Forest
W celu realizacji projektu zdecydowano się na algorytm Random Forest Regressor, należący do grupy metod zespołowych typu bagging. Model ten tworzy wiele drzew decyzyjnych (każde na losowej próbce bootstrapowej), a następnie wyznacza prognozę jako średnią z wyników uzyskanych przez poszczególne drzewa.

* **Parametry:** 500 drzew (`n_estimators=500`), maksymalna głębokość 25 oraz `min_samples_leaf=5` i `min_samples_split=20`, które ograniczają dopasowanie pojedynczych liści do unikalnych anomalii.

In [6]:
params = dict(n_estimators=500,     # liczba drzew 
    max_depth=25,                   # maksymalna głębokość drzewa
    min_samples_leaf=5,             # minimalna liczba próbek w liściu 
    min_samples_split=20,           # minimalna liczba próbek do podziału węzła
    max_features=1.0,               # liczba cech rozważanych przy każdym podziale; 1.0 = wszystkie cechy
    bootstrap=True,                 # czy każde drzewo ma być trenowane na próbce bootstrapowej
    random_state=42,                # losowość
    n_jobs=-1)                      # liczba rdzeni CPU; -1 = wykorzystanie wszystkich dostępnych rdzeni


### Trenowanie modelu i walidacja na danych z 2024 roku

Pierwszy etap obejmuje trenowanie modelu na danych historycznych z lat 2021–2023 oraz ocenę jego skuteczności na zbiorze walidacyjnym (rok 2024), na podstawie którego dobierano parametry modelu.
Eksperymentalnie wykazano, że Random Forest osiąga znacznie wyższą dokładność przy zastosowaniu **transformacji logarytmicznej zmiennej docelowej** (`np.log1p`).
Po wygenerowaniu prognozy wykonano transformację odwrotną za pomocą funkcji `expm1()`, przywracając wyniki do pierwotnej skali kilogramów sprzedaży (ujemne wartości obcinane są do 0).

In [7]:
# Model dla danych walidacyjnych 2024 
print("Trening na 2021–2023, walidacja na 2024:")

y_train_log = np.log1p(y_train)

model_v1 =  RandomForestRegressor(**params)

model_v1.fit(X_train, y_train_log)

pred_val_log = model_v1.predict(X_val)
pred_val = np.maximum(np.expm1(pred_val_log), 0)

# Ocena jakości modelu 
mae_val, rmse_val, r2_val, bias_val = print_metrics("WALIDACJA 2024", y_val, pred_val)

Trening na 2021–2023, walidacja na 2024:

────────────────────
  WALIDACJA 2024
────────────────────
MAE           : 1.3962 kg
RMSE          : 8.1246 kg
R²            : 0.3746
Bias          : -0.8180 kg
Średnia       : 1.6185 kg
MAE / średnia : 86.3%
WMAPE         : 74.2%


### Trening i wykonanie modelu końcowego
Dane treningowe oraz walidacyjne (2021–2024) zostały połączone w celu zwiększenia zakresu danych dla modelu końcowego. Zmienną docelową ponownie przekształcono logarytmicznie, a model wytrenowano na ustalonych hiperparametrach. Po predykcji odwrócono transformację, a prognozy poniżej 0,01 kg zaokrąglono do 0. Model oceniono na zbiorze testowym 2025, a prognozy dołączono do tabeli `test`.

In [10]:
# Model danych na 2021–2024 dla testu 2025

print("Model końcowy: Trening na danych 2021–2024, TEST na 2025")

X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

y_train_full_log = np.log1p(y_train_full)

# Model końcowy 
model_final = RandomForestRegressor(**params)
model_final.fit(X_train_full, y_train_full_log)

pred_test_log = model_final.predict(X_test)
pred_test = np.maximum(np.expm1(pred_test_log), 0)
pred_test = np.where(pred_test < 0.01, 0, pred_test)

# Ocena modelu
mae_test, rmse_test, r2_test, bias_test = print_metrics("TEST 2025", y_test, pred_test)

# Zapisanie wyników prognozy
test = X_test.copy()

test["Rzeczywista"] = y_test.values
test["Prognoza"] = pred_test

Model końcowy: Trening na danych 2021–2024, TEST na 2025

────────────────────
  TEST 2025
────────────────────
MAE           : 1.6098 kg
RMSE          : 9.2796 kg
R²            : 0.2540
Bias          : -1.0220 kg
Średnia       : 1.8155 kg
MAE / średnia : 88.7%
WMAPE         : 76.5%


### Analiza ważności cech
Wbudowana ważność cech (feature importance) modelu końcowego – 10 najważniejszych zmiennych. Model Random Forest opiera się głównie na `lag_364`, a następnie na podkategorii i kliencie.

In [13]:
rf_importance = pd.DataFrame({
    'Cecha': X_train_full.columns,
    'Ważność': model_final.feature_importances_
})

rf_importance = rf_importance.sort_values(
    by='Ważność',
    ascending=False
)

print('Analiza ważności cech dla Random Forest / Sklepy:')
print(rf_importance.head(10))

Analiza ważności cech dla Random Forest / Sklepy:
            Cecha   Ważność
19        lag_364  0.301011
1    Podkategoria  0.224615
0          Klient  0.137350
17  dni_do_swieta  0.082767
6         day_sin  0.071363
8    tydzien_roku  0.048373
9             rok  0.033814
7         day_cos  0.028754
10  czy_niedziela  0.017067
2     miesiac_sin  0.015470


### Obliczenie błędów prognozy i odkodowanie zmiennych tekstowych

Dla każdego wiersza liczony jest błąd (prognoza − rzeczywistość) oraz błąd bezwzględny, dodawane są etykiety modelu i segmentu, a kody klientów i podkategorii zamieniane są z powrotem na nazwy. Wynik zapisywany jest do `../wyniki/prognozy/prognozy_RF_2025.csv` w celu dalszej analizy w Power BI oraz porównania z XGBoost (notebook 06).

In [14]:
test["Błąd"] = test["Prognoza"] - test["Rzeczywista"]
test["Błąd_abs"] = abs(test["Błąd"])


test["Model"] = "Random_Forest"
test["Segment"] = "Sklepy"

# Przywrócenie nazw sklepów i podkategorii
test["Klient_nazwa"] = test["Klient"].map(mapowania["Klient"])
test["Podkategoria_nazwa"] = test["Podkategoria"].map(mapowania["Podkategoria"])
test.reset_index(inplace=True)

In [15]:
rf_sklep = test[['data_sprzedaży','Prognoza', 'Model',
       'Segment', 'Rzeczywista', 'Błąd', 'Błąd_abs', 'Klient_nazwa','Podkategoria_nazwa']]

In [16]:
rf_sklep.to_csv("../wyniki/prognozy/prognozy_RF_2025.csv", index=False, sep=";", decimal=",")